# Entrenamiento de CheXNet (DenseNet-121) sobre NIH ChestX-ray14

**Proyecto de Aula - Inteligencia Artificial - III Corte (Fase 3)**

Notebook disenado para ejecutarse en **Kaggle Notebooks** con GPU (Tesla T4 / P100).\nSe utiliza Kaggle porque la GPU local del equipo (AMD Radeon RX 580) no soporta CUDA.

## Configuracion esperada en Kaggle

1. Crear un nuevo notebook en Kaggle.
2. Settings -> Accelerator -> **GPU T4 x1** (o **P100**).
3. Add Data -> buscar y agregar:
   - `nih-chest-xrays/data` (dataset oficial, 45 GB).
   - Subir como dataset privado el archivo `data/processed/metadata_with_splits.csv` generado localmente con `build_splits.py`.

## Pipeline implementado

1. Configuracion global (rutas, hiperparametros).
2. Definicion inline del modelo, dataset y transforms (autocontenido).
3. Construccion de `DataLoaders` para train/val/test.
4. Entrenamiento con `BCEWithLogitsLoss(pos_weight=...)`, Adam, scheduler y mixed precision.
5. Evaluacion con AUC-ROC por clase, F1-Score y matriz de confusion multi-etiqueta.
6. Guardado del mejor modelo (mejor AUC en validacion) en `/kaggle/working/best_model.pt`.

## 1. Configuracion global

Las rutas estan adaptadas al sistema de archivos de Kaggle. Si el notebook se ejecuta
localmente, modifica las constantes de la clase `Config`.

In [ ]:
import os
import json
import time
import random
import logging
from pathlib import Path
from dataclasses import dataclass, field
from typing import List, Optional, Callable

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
import torchvision.models as models
import torchvision.transforms as T
from PIL import Image
from sklearn.metrics import roc_auc_score, f1_score, average_precision_score

print("PyTorch:", torch.__version__)
print("CUDA disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

ModuleNotFoundError: No module named 'pandas'

: 

In [ ]:
PATHOLOGIES = [
    "Atelectasis", "Cardiomegaly", "Effusion", "Infiltration", "Mass",
    "Nodule", "Pneumonia", "Pneumothorax", "Consolidation", "Edema",
    "Emphysema", "Fibrosis", "Pleural_Thickening", "Hernia",
]
N_CLASSES = len(PATHOLOGIES)


@dataclass
class Config:
    images_dir: str = "/kaggle/input/data"
    metadata_csv: str = "/kaggle/input/chestxray-splits/metadata_with_splits.csv"
    output_dir: str = "/kaggle/working"

    image_size: int = 224
    batch_size: int = 32
    num_workers: int = 2

    n_epochs: int = 10
    lr: float = 1e-4
    weight_decay: float = 1e-5
    dropout: float = 0.0

    use_amp: bool = True
    seed: int = 42

    log_every: int = 100

CFG = Config()
Path(CFG.output_dir).mkdir(parents=True, exist_ok=True)


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

set_seed(CFG.seed)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

## 2. Definiciones inline (autocontenidas)

En produccion estos componentes viven en `src/data/` y `src/models/`. Aqui los duplicamos
para que el notebook funcione en Kaggle sin tener que subir el repositorio completo.

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]


def get_train_transforms(image_size: int = 224) -> Callable:
    return T.Compose([
        T.Grayscale(num_output_channels=3),
        T.Resize((image_size, image_size), antialias=True),
        T.RandomAffine(degrees=7, translate=(0.05, 0.05), scale=(0.95, 1.05)),
        T.ColorJitter(brightness=0.1, contrast=0.1),
        T.ToTensor(),
        T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ])


def get_eval_transforms(image_size: int = 224) -> Callable:
    return T.Compose([
        T.Grayscale(num_output_channels=3),
        T.Resize((image_size, image_size), antialias=True),
        T.ToTensor(),
        T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ])


def build_image_index(images_root):
    """
    El dataset de Kaggle 'nih-chest-xrays/data' tiene las imagenes en 12 subcarpetas:
        images_001/images/00000001_000.png
        images_002/images/...
    Esta funcion construye un dict {image_id -> path_completo} explorando todas
    las subcarpetas una sola vez al inicio.
    """
    images_root = Path(images_root)
    index = {}
    direct = list(images_root.glob("*.png"))
    if direct:
        for p in direct:
            index[p.name] = p
        print(f"  Imagenes directas encontradas: {len(index):,}")
        return index
    for subdir in sorted(images_root.glob("images_*")):
        inner = subdir / "images"
        target = inner if inner.is_dir() else subdir
        for p in target.glob("*.png"):
            index[p.name] = p
    print(f"  Imagenes indexadas en subcarpetas: {len(index):,}")
    return index


class ChestXrayDataset(Dataset):
    def __init__(self, metadata_csv, images_dir, split, transform=None, image_index=None):
        df = pd.read_csv(metadata_csv)
        df = df[df["split"] == split].reset_index(drop=True)
        if df.empty:
            raise ValueError(f"split '{split}' vacio")
        self.df = df
        self.images_dir = Path(images_dir)
        self.transform = transform
        self.image_ids = df["image_id"].astype(str).tolist()
        self.labels = df[PATHOLOGIES].astype("float32").to_numpy()
        self.image_index = image_index if image_index is not None else build_image_index(images_dir)
        missing = [iid for iid in self.image_ids if iid not in self.image_index]
        if missing:
            print(f"  [WARN] {len(missing)} imagenes no encontradas en disco (de {len(self.image_ids):,})")
        print(f"  [{split}] {len(df):,} imgs  {df['patient_id'].nunique():,} pacientes")

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        image_id = self.image_ids[idx]
        path = self.image_index.get(image_id)
        try:
            if path is None:
                raise FileNotFoundError(image_id)
            img = Image.open(path).convert("L")
        except (FileNotFoundError, OSError):
            img = Image.new("L", (224, 224), color=0)
        if self.transform is not None:
            img = self.transform(img)
        label = torch.from_numpy(self.labels[idx])
        return img, label


def compute_pos_weight(metadata_csv, split="train"):
    df = pd.read_csv(metadata_csv)
    df = df[df["split"] == split]
    weights = []
    for patho in PATHOLOGIES:
        pos = float(df[patho].sum())
        neg = float(len(df) - pos)
        weights.append(neg / pos if pos > 0 else 1.0)
    return torch.tensor(weights, dtype=torch.float32)

In [ ]:
class DenseNetCheXNet(nn.Module):
    """DenseNet-121 con cabeza Linear(1024 -> 14). Devuelve logits (sin sigmoid)."""

    def __init__(self, n_classes=N_CLASSES, pretrained=True, dropout=0.0):
        super().__init__()
        weights = models.DenseNet121_Weights.IMAGENET1K_V1 if pretrained else None
        backbone = models.densenet121(weights=weights)
        in_features = backbone.classifier.in_features
        if dropout > 0.0:
            backbone.classifier = nn.Sequential(
                nn.Dropout(p=dropout),
                nn.Linear(in_features, n_classes),
            )
            head = backbone.classifier[1]
        else:
            backbone.classifier = nn.Linear(in_features, n_classes)
            head = backbone.classifier
        nn.init.xavier_uniform_(head.weight)
        nn.init.zeros_(head.bias)
        self.backbone = backbone

    def forward(self, x):
        return self.backbone(x)

## 3. Construccion de datasets y dataloaders

In [ ]:
print("Indexando imagenes (una sola vez)...")
image_index = build_image_index(CFG.images_dir)

print("\nConstruyendo datasets...")
train_ds = ChestXrayDataset(CFG.metadata_csv, CFG.images_dir, "train",
                             transform=get_train_transforms(CFG.image_size),
                             image_index=image_index)
val_ds   = ChestXrayDataset(CFG.metadata_csv, CFG.images_dir, "val",
                             transform=get_eval_transforms(CFG.image_size),
                             image_index=image_index)
test_ds  = ChestXrayDataset(CFG.metadata_csv, CFG.images_dir, "test",
                             transform=get_eval_transforms(CFG.image_size),
                             image_index=image_index)

train_loader = DataLoader(train_ds, batch_size=CFG.batch_size, shuffle=True,
                          num_workers=CFG.num_workers, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds, batch_size=CFG.batch_size, shuffle=False,
                          num_workers=CFG.num_workers, pin_memory=True)
test_loader  = DataLoader(test_ds, batch_size=CFG.batch_size, shuffle=False,
                          num_workers=CFG.num_workers, pin_memory=True)

print(f"Batches: train={len(train_loader)}  val={len(val_loader)}  test={len(test_loader)}")

## 4. Modelo, loss, optimizador

- **Modelo**: DenseNet-121 con cabeza Linear(1024 -> 14).
- **Loss**: `BCEWithLogitsLoss(pos_weight=...)` para compensar el desbalance extremo
  (Infiltration es 87x mas frecuente que Hernia en train).
- **Optimizador**: Adam con `weight_decay=1e-5`.
- **Scheduler**: ReduceLROnPlateau monitoreando AUC promedio en validacion.

In [ ]:
model = DenseNetCheXNet(n_classes=N_CLASSES, pretrained=True, dropout=CFG.dropout).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"Modelo: DenseNetCheXNet  -  {n_params:,} parametros")

pos_weight = compute_pos_weight(CFG.metadata_csv, split="train").to(device)
print("\npos_weight por clase:")
for patho, w in zip(PATHOLOGIES, pos_weight.cpu().tolist()):
    print(f"  {patho:<22} {w:>8.2f}")

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(model.parameters(), lr=CFG.lr, weight_decay=CFG.weight_decay)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.1, patience=2,
)
scaler = GradScaler(enabled=CFG.use_amp and torch.cuda.is_available())

## 5. Funciones de entrenamiento y evaluacion

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, scaler, device, log_every=100, epoch=0):
    model.train()
    losses = []
    t0 = time.time()
    for step, (images, labels) in enumerate(loader):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        with autocast(enabled=scaler.is_enabled()):
            logits = model(images)
            loss = criterion(logits, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        losses.append(loss.item())

        if (step + 1) % log_every == 0:
            elapsed = time.time() - t0
            print(f"  ep {epoch} step {step+1}/{len(loader)}  loss={np.mean(losses[-log_every:]):.4f}  ({elapsed:.0f}s)")
    return float(np.mean(losses))


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    all_logits, all_labels, all_losses = [], [], []
    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        with autocast(enabled=torch.cuda.is_available() and CFG.use_amp):
            logits = model(images)
            loss = criterion(logits, labels)
        all_losses.append(loss.item())
        all_logits.append(logits.float().cpu().numpy())
        all_labels.append(labels.cpu().numpy())

    logits = np.concatenate(all_logits, axis=0)
    labels = np.concatenate(all_labels, axis=0)
    probs = 1.0 / (1.0 + np.exp(-logits))

    aucs, f1s, aps = [], [], []
    for i, patho in enumerate(PATHOLOGIES):
        if labels[:, i].sum() == 0 or labels[:, i].sum() == len(labels):
            aucs.append(np.nan); f1s.append(np.nan); aps.append(np.nan)
            continue
        aucs.append(roc_auc_score(labels[:, i], probs[:, i]))
        f1s.append(f1_score(labels[:, i], (probs[:, i] >= 0.5).astype(int), zero_division=0))
        aps.append(average_precision_score(labels[:, i], probs[:, i]))

    return {
        "loss": float(np.mean(all_losses)),
        "auc_mean": float(np.nanmean(aucs)),
        "auc_per_class": dict(zip(PATHOLOGIES, aucs)),
        "f1_per_class": dict(zip(PATHOLOGIES, f1s)),
        "ap_per_class": dict(zip(PATHOLOGIES, aps)),
        "probs": probs,
        "labels": labels,
    }

## 6. Loop principal de entrenamiento

Se guarda el mejor checkpoint segun AUC promedio en validacion (`best_model.pt`).

In [ ]:
history = []
best_auc = -1.0
best_path = Path(CFG.output_dir) / "best_model.pt"

for epoch in range(1, CFG.n_epochs + 1):
    print(f"\n========== EPOCH {epoch}/{CFG.n_epochs} ==========")
    t0 = time.time()
    train_loss = train_one_epoch(
        model, train_loader, criterion, optimizer, scaler, device,
        log_every=CFG.log_every, epoch=epoch,
    )
    val_metrics = evaluate(model, val_loader, criterion, device)
    scheduler.step(val_metrics["auc_mean"])

    epoch_time = time.time() - t0
    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_metrics["loss"],
        "val_auc_mean": val_metrics["auc_mean"],
        "lr": optimizer.param_groups[0]["lr"],
        "time_sec": epoch_time,
    })

    print(f"\n  train_loss={train_loss:.4f}  val_loss={val_metrics['loss']:.4f}  "
          f"val_AUC={val_metrics['auc_mean']:.4f}  lr={optimizer.param_groups[0]['lr']:.2e}  "
          f"time={epoch_time:.0f}s")
    print("  AUC por clase:")
    for patho, auc in val_metrics["auc_per_class"].items():
        print(f"    {patho:<22} AUC={auc:.4f}")

    if val_metrics["auc_mean"] > best_auc:
        best_auc = val_metrics["auc_mean"]
        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "val_auc": best_auc,
            "config": CFG.__dict__,
        }, best_path)
        print(f"  -> Nuevo best_AUC={best_auc:.4f}  (guardado en {best_path})")

with open(Path(CFG.output_dir) / "history.json", "w") as f:
    json.dump(history, f, indent=2)
print(f"\nFin. Best val_AUC={best_auc:.4f}")

## 7. Evaluacion final sobre el conjunto de TEST oficial del NIH

Se carga el mejor checkpoint y se evalua sobre las 25,596 imagenes de test.

**Recordar (seccion 7.2 del EDA)**: el test set del NIH esta enriquecido en casos
enfermos (todas las patologias son mas prevalentes que en train_val), por lo que
las metricas seran INFERIORES a las de validacion. Esto es esperado.

In [ ]:
checkpoint = torch.load(best_path, map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])
print(f"Cargado checkpoint epoca {checkpoint['epoch']}, val_AUC={checkpoint['val_auc']:.4f}")

print("\nEvaluando en TEST...")
test_metrics = evaluate(model, test_loader, criterion, device)

print(f"\n=== TEST RESULTS ===")
print(f"  loss      : {test_metrics['loss']:.4f}")
print(f"  AUC mean  : {test_metrics['auc_mean']:.4f}")
print(f"  AUC por clase:")
for patho in PATHOLOGIES:
    auc = test_metrics["auc_per_class"][patho]
    f1  = test_metrics["f1_per_class"][patho]
    ap  = test_metrics["ap_per_class"][patho]
    print(f"    {patho:<22} AUC={auc:.4f}  F1={f1:.4f}  AP={ap:.4f}")

results = {
    "test_loss": test_metrics["loss"],
    "test_auc_mean": test_metrics["auc_mean"],
    "test_auc_per_class": test_metrics["auc_per_class"],
    "test_f1_per_class": test_metrics["f1_per_class"],
    "test_ap_per_class": test_metrics["ap_per_class"],
    "best_val_auc": float(checkpoint["val_auc"]),
    "best_epoch": int(checkpoint["epoch"]),
    "config": CFG.__dict__,
}
with open(Path(CFG.output_dir) / "test_results.json", "w") as f:
    json.dump(results, f, indent=2)

np.savez_compressed(
    Path(CFG.output_dir) / "test_predictions.npz",
    probs=test_metrics["probs"],
    labels=test_metrics["labels"],
    image_ids=np.array(test_ds.image_ids),
)
print(f"\nResultados guardados en {CFG.output_dir}")

## 8. Curvas de entrenamiento

Visualizacion rapida de loss y AUC a lo largo de las epocas para verificar la convergencia.

In [ ]:
import matplotlib.pyplot as plt

epochs = [h["epoch"] for h in history]
train_losses = [h["train_loss"] for h in history]
val_losses = [h["val_loss"] for h in history]
val_aucs = [h["val_auc_mean"] for h in history]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(epochs, train_losses, "o-", label="train")
axes[0].plot(epochs, val_losses, "s-", label="val")
axes[0].set_xlabel("Epoca"); axes[0].set_ylabel("Loss")
axes[0].set_title("Loss por epoca")
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(epochs, val_aucs, "o-", color="green")
axes[1].set_xlabel("Epoca"); axes[1].set_ylabel("AUC promedio (val)")
axes[1].set_title("AUC-ROC promedio en validacion")
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(Path(CFG.output_dir) / "training_curves.png", dpi=120, bbox_inches="tight")
plt.show()